# 🌐 TD08 — Yahoo Finance Market Data Provider via yfinance
## MarketPulse — Feature 2: `feature/yahoo-provider` ➔ `src/providers/yahoo_provider.py`

In this notebook, we build the remote market data adapter for **Yahoo Finance**.

---
### 👥 Team Organization (4-Student Group)
* **Assigned Developer (PR Author)**: **Student 3** (Lead Yahoo Provider)
* **Official Code Reviewer**: **Student 4**
* **Dedicated Git Branch**: `feature/yahoo-provider`
* **Assessment Milestone**: Completes **Checkpoint B (30%)**
---

### 🎯 Notebook Objectives:
1. Discover remote historical market data acquisition using **`yfinance`**.
2. Map internal canonical tickers (e.g., `SP500`) to Yahoo Finance symbols (e.g., `^GSPC`).
3. Transform raw yfinance history into the **MarketPulse canonical row contract**:
   ```python
   {
       "date": "YYYY-MM-DD",
       "ticker": "AAPL",     # Internal canonical ticker
       "open": 180.25,       # float
       "high": 182.10,       # float
       "low": 179.80,        # float
       "close": 181.50,      # float
       "volume": 52400000    # int
   }
   ```
4. Follow the **Git Integration Guide** at the end of this notebook to create `src/providers/yahoo_provider.py`.


## 1. Symbol Mapping and Acquisition Function


In [ ]:
import yfinance as yf

# Mapping between MarketPulse canonical tickers and Yahoo Finance symbols
YAHOO_SYMBOL_MAPPING = {
    "AAPL": "AAPL",
    "SP500": "^GSPC",
}


def fetch_yahoo_prices(
    symbol,
    canonical_ticker,
    period="1mo",
    interval="1d",
):
    """
    Retrieve historical market prices from Yahoo Finance and normalize to canonical format.
    
    Args:
        symbol (str): Yahoo Finance symbol (e.g., 'AAPL' or '^GSPC').
        canonical_ticker (str): Internal canonical ticker (e.g., 'AAPL' or 'SP500').
        period (str): Historical lookback window (default: '1mo').
        interval (str): Price frequency (default: '1d').
        
    Returns:
        list: Canonical observation dictionaries dict(date, ticker, open, high, low, close, volume).
    """
    history = yf.Ticker(symbol).history(
        period=period,
        interval=interval,
        auto_adjust=False,
    )

    if history.empty:
        raise ValueError(
            f"No Yahoo Finance data returned for symbol {symbol} "
            f"(period={period}, interval={interval})."
        )

    rows = []
    for timestamp, row in history.iterrows():
        rows.append({
            "date": timestamp.strftime("%Y-%m-%d"),
            "ticker": canonical_ticker,
            "open": float(row["Open"]),
            "high": float(row["High"]),
            "low": float(row["Low"]),
            "close": float(row["Close"]),
            "volume": int(row["Volume"]),
        })

    return rows


## 2. Interactive Acquisition and Normalization Test


In [ ]:
# Test acquisition for AAPL
symbol_aapl = YAHOO_SYMBOL_MAPPING["AAPL"]
aapl_canonical = fetch_yahoo_prices(symbol_aapl, canonical_ticker="AAPL", period="1mo", interval="1d")

print(f"Number of AAPL observations retrieved: {len(aapl_canonical)}")
print("First canonical observation sample:")
print(aapl_canonical[0])

# Test acquisition for S&P 500 (^GSPC)
symbol_sp500 = YAHOO_SYMBOL_MAPPING["SP500"]
sp500_canonical = fetch_yahoo_prices(symbol_sp500, canonical_ticker="SP500", period="1mo", interval="1d")

print(f"\nNumber of SP500 observations retrieved: {len(sp500_canonical)}")
print("First canonical observation sample:")
print(sp500_canonical[0])


## 3. Strict Type Verification on Canonical Rows


In [ ]:
sample_row = aapl_canonical[0]

assert isinstance(sample_row["date"], str), "Date must be a 'YYYY-MM-DD' formatted string"
assert isinstance(sample_row["ticker"], str), "Ticker must be a string"
assert isinstance(sample_row["open"], float), "Open price must be float"
assert isinstance(sample_row["high"], float), "High price must be float"
assert isinstance(sample_row["low"], float), "Low price must be float"
assert isinstance(sample_row["close"], float), "Close price must be float"
assert isinstance(sample_row["volume"], int), "Volume must be int"

print("✅ All field types strictly conform to the MarketPulse canonical contract!")


---
## 🛠️ Step-by-Step Git Integration Guide (Team of 4)

Follow these steps to integrate Feature 2 into your repository:

### Step 1: Branch Preparation (by Student 3)
```bash
git switch main
git pull origin main
git switch -c feature/yahoo-provider
```

### Step 2: Create the Providers Package and Module
1. Ensure the directory `src/providers/` exists.
2. Create an empty package initialization file **`src/providers/__init__.py`**:
   ```bash
   touch src/providers/__init__.py
   ```
3. Create **`src/providers/yahoo_provider.py`** and copy:
   - The import: `import yfinance as yf`
   - The mapping: `YAHOO_SYMBOL_MAPPING`
   - The function: `fetch_yahoo_prices(...)`

### Step 3: Local Verification
```bash
python3 -c "from src.providers.yahoo_provider import fetch_yahoo_prices; print('✅ yahoo_provider.py is ready!')"
```

### Step 4: Commit and Push
```bash
git add src/providers/
git commit -m "feat(provider): implement remote yahoo finance data adapter"
git push -u origin feature/yahoo-provider
```

### Step 5: Pull Request & Checkpoint B Validation
- Student 3 opens the PR `feature/yahoo-provider` ➔ `main`.
- **Reviewer**: Student 4.
- 📸 **Checkpoint B Evidence (Student 3)**: PR screenshot for `01_pull_request.png`.
- 📸 **Checkpoint B Evidence (Students 3 & 4)**: Remote retrieval screenshots for `03_yahoo_market_data.png` and `04_instrument_benchmark.png`.
- Student 4 reviews in `Files changed`, posts constructive feedback (`02_code_review.png`), approves and merges.
